In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from Bio import SeqIO
import os
from collections import Counter

import statsmodels.stats.multitest 

In [3]:
base_dir = '/Users/cdubin/VMGC_cervical_dysplasia_paper/code/'

In [ ]:
species = pd.read_csv(f'{base_dir}/compare_VMGC_GTDB/select_species_for_analysis/shared_species_for_analysis.csv')
species = species.set_index('species_id_VMGC')
species_list = species.index.tolist()

In [14]:
species_list

[988598, 240891, 783244, 619501, 571325, 611554]

In [5]:
def is_string_in_list(target, joined_list):
    if pd.isna(joined_list):
        return False
    return target in joined_list.split(',')

def get_enrichment(pfam, test_df, df, resample_count=1000):
    
    num_genes_to_resample = test_df.shape[0]
    real_count = test_df[test_df['PFAMs'].apply(lambda x: is_string_in_list(pfam, x))].shape[0]
    higher_count = 0
    
    for _ in range(resample_count):
        
        t = df.sample(num_genes_to_resample)
        sample_count = t[t['PFAMs'].apply(lambda x: is_string_in_list(pfam, x))].shape[0]
        if sample_count >= real_count:
            higher_count+=1
            
    return (higher_count+1)/(resample_count+1)

In [6]:
!mkdir PFAM_analysis

In [15]:
np.random.seed(10)
results = {}
for sp in [240891,988598,571325,619501,611554,783244]:        
        
    results[sp] = {}
    to_concat = []

    outpath = f'PFAM_analysis/{sp}_PFAM_enrichment_C90.csv'

    # if os.path.exists(outpath):
        # print(f'Output found: {outpath}')
        # continue

    ## load data
    c90_classified= pd.read_csv(f'centroid_classification/{sp}_C90_classified.csv', index_col=0)
    c90_info = pd.read_csv(f'{base_dir}/compare_VMGC_GTDB/combined_db/VMGC_GTDB_combined_db/pangenomes/{sp}/clusters_90_info.tsv', sep='\t', index_col=0)
    c90_info = c90_info.merge(c90_classified, left_index=True, right_index=True)

    ## get list of pfams found in vmgc and gtdb exclusive centroids
    vmgc_pfams = []
    for p in c90_info[c90_info['classification'] == 'VMGC']['PFAMs'].tolist():
        if pd.isna(p) or p == '-':
            continue
        vmgc_pfams += p.split(',')    

    gtdb_pfams = []
    for p in c90_info[c90_info['classification'] == 'GTDB']['PFAMs'].tolist():
        if pd.isna(p) or p == '-':
            continue
        gtdb_pfams += p.split(',')   

    # test for enrichment
    for grouping, pfam_list in zip(['VMGC', 'GTDB'], [vmgc_pfams, gtdb_pfams]):

        counts = Counter(pfam_list)
        p_temp = []
        
        print(f'starting analysis of {len(set(counts))} PFAMs for {grouping} centroids in {sp}')
        
        if len(counts) == 0:
            print(f'No {grouping} PFAMs found for {sp}')
            continue
            
        for i, (pfam, count) in enumerate(dict(counts).items()):

            if count <= 9:
                continue
            p = get_enrichment(pfam, c90_info[c90_info['classification'] == grouping], c90_info)
            p_temp += [[pfam, count, p]]
    
        
        if len(p_temp) == 0:
            print(f'No non-singleton {grouping} PFAMs found for {sp}')
            continue
            
        p_res = pd.DataFrame(p_temp)
        p_res.columns = ['PFAM','count','p']
        p_res = p_res.sort_values('p')

        p_res['enrichment'] = grouping
        p_res['species'] = sp
        to_concat += [p_res]

    p_res = pd.concat(to_concat)
    p_res['p_BHadj'] = statsmodels.stats.multitest.multipletests(p_res['p'].values, method='fdr_bh')[1]
    p_res = p_res.sort_values('p_BHadj')
    results[sp] = p_res

    p_res.to_csv(outpath)

    for grouping in ['VMGC', 'GTDB']:

        temp = p_res[p_res['enrichment'] == grouping]
        print(f'{(temp["p_BHadj"] < 0.05).sum()} signficant PFAMs for {grouping} exclusive centroids in {sp}')
        
    print()

starting analysis of 1074 PFAMs for VMGC centroids in 240891
starting analysis of 0 PFAMs for GTDB centroids in 240891
No GTDB PFAMs found for 240891
0 signficant PFAMs for VMGC exclusive centroids in 240891
0 signficant PFAMs for GTDB exclusive centroids in 240891

starting analysis of 1041 PFAMs for VMGC centroids in 988598
starting analysis of 703 PFAMs for GTDB centroids in 988598
3 signficant PFAMs for VMGC exclusive centroids in 988598
11 signficant PFAMs for GTDB exclusive centroids in 988598

starting analysis of 597 PFAMs for VMGC centroids in 571325
starting analysis of 42 PFAMs for GTDB centroids in 571325
No non-singleton GTDB PFAMs found for 571325
0 signficant PFAMs for VMGC exclusive centroids in 571325
0 signficant PFAMs for GTDB exclusive centroids in 571325

starting analysis of 918 PFAMs for VMGC centroids in 619501
starting analysis of 0 PFAMs for GTDB centroids in 619501
No GTDB PFAMs found for 619501
2 signficant PFAMs for VMGC exclusive centroids in 619501
0 sign

In [25]:
g = pd.read_csv('PFAM_analysis/988598_PFAM_enrichment_C90.csv', index_col=0)
g['p_BHadj'] = g['p_BHadj'].round(4)
g[(g['p_BHadj'] < 0.05) & (g['enrichment'] == 'VMGC')]

,PFAM,count,p,enrichment,species,p_BHadj
32,Bro-N,14,0.000999,VMGC,988598,0.0080
13,AAA_24,11,0.006993,VMGC,988598,0.0356
3,Terminase_1,12,0.009990,VMGC,988598,0.0430


In [30]:
g = pd.read_csv('PFAM_analysis/988598_PFAM_enrichment_C90.csv', index_col=0)
g['p_BHadj'] = g['p_BHadj'].round(4)
g[(g['p_BHadj'] < 0.05) ]

,PFAM,count,p,enrichment,species,p_BHadj
32,Bro-N,14,0.000999,VMGC,988598,0.0080
8,Glycos_transf_2,40,0.000999,GTDB,988598,0.0080
7,AbiH,10,0.000999,GTDB,988598,0.0080
2,YSIRK_signal,42,0.000999,GTDB,988598,0.0080
1,Gram_pos_anchor,43,0.000999,GTDB,988598,0.0080
20,Methylase_S,41,0.000999,GTDB,988598,0.0080
11,Glycos_transf_1,37,0.000999,GTDB,988598,0.0080
18,N6_Mtase,19,0.001998,GTDB,988598,0.0140
15,Polysacc_synt_C,15,0.003996,GTDB,988598,0.0224
16,Glyco_transf_4,16,0.003996,GTDB,988598,0.0224


In [17]:
g = results[619501]
g[g['p_BHadj'] < 0.05]

,PFAM,count,p,enrichment,species,p_BHadj
24,CHAP,53,0.000999,VMGC,619501,0.018981
8,Methylase_S,124,0.000999,VMGC,619501,0.018981
